In [ ]:
%load_ext autoreload
%autoreload 2


# Rule-Based Model Example

This notebook shows how to run the packaged methylseg pathway using the rule-based state definitions before HMM segmentation.


In [ ]:
from pathlib import Path
import pandas as pd

from methylseg import (
    MethylDataPrep,
    MethylSegPathway,
    MethylStateAssignmentMethod,
    MethylationStates,
)


In [ ]:
ip = get_ipython()
if "__vsc_ipynb_file__" in ip.user_ns:
    EXAMPLES_DIR = Path(ip.user_ns["__vsc_ipynb_file__"]).resolve().parent
else:
    EXAMPLES_DIR = Path.cwd()

PACKAGE_ROOT = EXAMPLES_DIR.parent
REFERENCE_DIR = PACKAGE_ROOT / "data" / "reference_files"

OUT_DIR = EXAMPLES_DIR / "out" / "rule_based_model_output"


In [ ]:
model = MethylSegPathway.get_pretrained_model(OUT_DIR, resolution="450k")
model.state_assignment_method = MethylStateAssignmentMethod.DEFINITION
model.segmentor.state_assignment_method = MethylStateAssignmentMethod.DEFINITION


In [ ]:
sample_info, sample_info_removed = MethylDataPrep(
    meth_file=REFERENCE_DIR / "TCGA-BD-A3EP-01A_450k.tsv.gz",
    sample_id="TCGA-BD-A3EP-01A",
    resolution="450k",
    remove_low_coverage_like_cpgs=True,
).prepare()

sample_info.sample_id


In [ ]:
model.analyzer.pretty_print_rules()


In [ ]:
regions_chr1 = model.generate_regions(sample_info=sample_info, chrom="chr1")
regions_chr1.head()


In [ ]:
fig = model.plot_labels(
    label_source="rule_based",
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom="chr1",
    label_title="Rule-based state",
)


In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    sample_info_removed=sample_info_removed,
    chrom="chr1",
    label_title="HMM state",
)


In [ ]:
clean_pmd_chr1 = model.get_clean_regions(
    regions_df=regions_chr1,
    state="PMD",
    sample_id=sample_info.sample_id,
    chrom="chr1",
)
clean_pmd_chr1.head()


In [ ]:
fig = model.plot_labels(
    label_source="hmm",
    sample_info=sample_info,
    chrom="chr1",
    clean_regions=True,
    overlay_state="PMD",
    region_start=2_000_000,
    region_end=4_000_000,
    region_chrom="chr1",
    label_title="HMM state",
)


In [ ]:
region_paths = model.run_pathway(
    sample_info=sample_info,
    chroms=["chr1"],
    clean_regions=True,
)
region_paths


In [ ]:
pd.read_csv(region_paths[1], sep="\t", header=None).head()
